#Initialization

In [0]:
%sql
USE CATALOG workspace

# Read Bronze table

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df AS
SELECT * FROM workspace.bronze.crm_prd_info

#Silver Transformations

##Trimming

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df_trimmed AS
SELECT prd_id, TRIM(prd_key) AS prd_key, TRIM(prd_nm) AS prd_nm, prd_cost,
       TRIM(prd_line) AS prd_line, prd_start_dt, prd_end_dt
FROM df

##Product Key Parsing

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df_parsed AS
SELECT prd_id, SUBSTRING(prd_key, 7, LENGTH(prd_key)) AS prd_key, prd_nm, prd_cost,
       prd_line, prd_start_dt, prd_end_dt,
       REGEXP_REPLACE(SUBSTRING(prd_key, 1, 5), '-', '_') AS cat_id
FROM df_trimmed

##Cost Cleanup

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df_cost AS
SELECT prd_id, prd_key, prd_nm, COALESCE(prd_cost, 0) AS prd_cost,
       prd_line, prd_start_dt, prd_end_dt, cat_id
FROM df_parsed

##Product Line Normalization

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df_line AS
SELECT prd_id, prd_key, prd_nm, prd_cost,
  CASE WHEN UPPER(prd_line) = 'M' THEN 'Mountain'
       WHEN UPPER(prd_line) = 'R' THEN 'Road'
       WHEN UPPER(prd_line) = 'S' THEN 'Other Sales'
       WHEN UPPER(prd_line) = 'T' THEN 'Touring'
       ELSE 'n/a' END AS prd_line,
  prd_start_dt, prd_end_dt, cat_id
FROM df_cost

## Date Casting

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df_dates AS
SELECT prd_id, prd_key, prd_nm, prd_cost, prd_line,
       CAST(prd_start_dt AS DATE) AS prd_start_dt, prd_end_dt, cat_id
FROM df_line

## Renaming Columns

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df_renamed AS
SELECT prd_id AS product_id, prd_key AS product_number, prd_nm AS product_name,
       prd_cost AS product_cost, prd_line AS product_line, prd_start_dt AS start_date,
       prd_end_dt AS end_date, cat_id AS category_id
FROM df_dates

## Sanity checks of dataframe

In [0]:
%sql
SELECT * FROM df_renamed LIMIT 10

#Writing Silver Table

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.crm_products USING DELTA AS
SELECT * FROM df_renamed

## Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.crm_products LIMIT 10